# Lab 1 – 在 Wikipedia 上進行接續預訓練（Continued Pretraining）
**模型：** `nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16` &nbsp;|&nbsp; **資料：** WikiText-103 &nbsp;|&nbsp; **訓練步數：** 10

流程：  
1. 下載 WikiText-103 → JSONL → **Megatron 前處理**  
2. 執行 10 步的接續預訓練


## 0. 環境確認

In [ ]:
import os, sys
from datasets import load_dataset
import json

print("Python :", sys.version.split()[0])
print("GPU    :", end=" ")
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import nemo_automodel
print("NeMo Automodel:", nemo_automodel.__version__)

## 1. 下載 WikiText-103 並檢視 JSONL 格式

In [ ]:
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# 下載一小部分資料 – 足夠進行 10 步訓練
print("正在下載 WikiText-103-raw-v1 …")
wiki = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:5000]")
print(f"載入了 {len(wiki)} 筆樣本")


In [ ]:
WIKI_JSONL = DATA_DIR / "wiki.jsonl"
kept = 0
with open(WIKI_JSONL, "w") as f:
    for item in wiki:
        text = item["text"].strip()
        if len(text) > 200:                        # 過濾掉章節標題和空行
            json.dump({"text": text}, f)
            f.write("\n")
            kept += 1
print(f"已寫入 {kept} 篇文件 → {WIKI_JSONL}  ({WIKI_JSONL.stat().st_size // 1024} KB)")


In [ ]:
# ── JSONL 結構 ────────────────────────────────────────────────────────────────
print("JSONL 格式：每一行是一個 JSON 物件，key = \"text\"")
print("─" * 60)
with open(WIKI_JSONL) as f:
    for i, line in enumerate(f):
        if i == 3: break
        print(line)


## 2. Megatron 前處理

`preprocess_megatron_dataset.py` 會將 JSONL 語料 tokenize 後寫成
memory-mapped 的二進位檔（`.bin` + `.idx`），Megatron 在訓練時可高速讀取。

重要參數：
```
--input                          輸入 JSONL 檔的路徑
--output-prefix / --output-path  輸出 .bin / .idx 檔的位置
--pretrained-model-name-or-path  使用的 tokenizer
--append-eod                     在文件結尾加上 end-of-document token
```

> 註：26.04 容器內的工具沒對 `AutoTokenizer.from_pretrained` 傳 `trust_remote_code`，
> 因此下面的 cell 會先用 sed 把這旗標加進工具中再執行（每個新 container 都要做一次）。


In [ ]:
# Megatron 預處理工具預設沒對 AutoTokenizer 傳 trust_remote_code，
# Nemotron-3 的 config.json 用內建（舊版）transformers nemotron_h 解析時會丟 KeyError: '-'。
# 在容器內用 sed 把工具的三處 from_pretrained 加上 trust_remote_code=True。
!sed -i 's|AutoTokenizer.from_pretrained(self.args.pretrained_model_name_or_path)|AutoTokenizer.from_pretrained(self.args.pretrained_model_name_or_path, trust_remote_code=True)|g' /opt/Automodel/tools/preprocess_megatron_dataset.py

!python /opt/Automodel/tools/preprocess_megatron_dataset.py \
    --input                          data/wiki.jsonl \
    --output-prefix                  wiki \
    --output-path                    data/wiki_megatron/ \
    --pretrained-model-name-or-path  nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 \
    --append-eod \
    --workers                        2


In [ ]:
!ls -lh data/wiki_megatron/

## 3. 預訓練設定檔

基礎的 YAML 設定（`pretrain_nemotron3_wiki.yaml`）放在這個 notebook 旁邊。
為了讓單一執行的設定集中、可控，幾個與「本次執行」相關的欄位
會在指令列上以參數覆寫：

| 覆寫欄位 | 值 | 原因 |
|---|---|---|
| `step_scheduler.max_steps` | 10 | 工作坊示範用 |
| `step_scheduler.val_every_steps` | 1000 | 跳過 validation |
| `step_scheduler.ckpt_every_steps` | 1000 | 跳過 checkpoint |


## 4. 執行預訓練（10 步）

In [ ]:
!torchrun --nproc-per-node=1 \
    /opt/Automodel/examples/llm_pretrain/pretrain.py \
    --config pretrain_nemotron3_wiki.yaml \
    --model.pretrained_model_name_or_path /workspace/nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16 \
    --step_scheduler.max_steps=10 \
    --step_scheduler.val_every_steps=10 \
    --step_scheduler.ckpt_every_steps=10 \
    --checkpoint.enabled=true \
    --checkpoint.checkpoint_dir=./checkpoints/pretrain/ \
    --checkpoint.model_save_format=safetensors \
    --checkpoint.save_consolidated=true


## 總結

| 階段 | 做了什麼 |
|---|---|
| JSONL | `{"text": "…"}` – 每行一篇 Wikipedia 文件 |
| Megatron 前處理 | Tokenize 後產生 `.bin` + `.idx` memory-mapped 檔 |
| 預訓練 | 在 wiki 資料上對 Nemotron-3-Nano-4B 進行 10 步接續預訓練 |

→ 請繼續前往 **Lab 2** 進行 SFT 與 LoRA PEFT。


## 5. 釋放磁碟空間

刪除本實驗產生的預處理後的 Wiki 資料（`data/`）
若還想複習結果可暫時略過；確認不再使用後再執行。


In [ ]:
import shutil

shutil.rmtree("data")